### Phase 1: Datenbeschaffung & Preprocessing
Zuerst benötigen wir die Rohdaten. Der Zeitraum von mindestens 20-30 Jahren ist ideal (um mehrere Zyklen wie 2000, 2008 und 2020 abzudecken).

*   **Libraries:** `yfinance`, `pandas`, `numpy`
*   **Datenquellen:**
    *   **Risiko-Asset:** 60% S&P 500 (Ticker: `^GSPC`), 40% US-Staatsanleihen (Ticker: `VUSTX`)
    *   **Safe-Haven:** 3-Monats-Treasury Bill Rate als Proxy für Cash-Zinsen(Ticker: `^IRX`)
    *   **Features:** VIX Index (Volatilität), Renditestrukturkurve (10Y-2Y Spread), Momentum-Indikatoren.

In [ ]:
# --- Central Config ---
import sys; sys.path.insert(0, "../config")
from config_loader import cfg

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# 1. Daten laden
# ^GSPC = S&P 500 | VUSTX = Long Bonds | ^VIX = Volatilität | ^IRX = 3-Monats-Zins | ^TNX = 10-Jahres-Zins
tickers = cfg.data.tickers
start_date = cfg.data.start_date
# Enddatum auf HEUTE setzen (yfinance lädt exklusive Enddatum, d.h. bis gestern inklusive)
end_date = cfg.data.end_date
#end_date = "2025-12-31"

# Alles ohne Index-Zugriff runterladen
raw_data = yf.download(tickers, start=start_date, end=end_date)

# --- ROBUSTER MULTI-INDEX FIX (Keyerror-Fix) ---
# Wir prüfen, welcher Preis-Typ verfügbar ist ('Adj Close' bevorzugt, sonst 'Close') - In neueren yfinance-Versionen wurde 'Adj Close' oft durch 'Close' ersetzt, wenn die Daten bereits bereinigt sind. Prüfung beider Case
if 'Adj Close' in raw_data.columns.get_level_values(0):
    data = raw_data['Adj Close'].copy()
    print("Nutze 'Adj Close'")
else:
    data = raw_data['Close'].copy()
    print("Nutze 'Close' (Adj Close nicht gefunden)")
    
# Füllen der Lücken beim Zins (^IRX), da dieser seltener schwankt
data['^IRX'] = data['^IRX'].ffill()
data = data.dropna()

# Jetzt haben wir ein "flaches" DataFrame mit den Tickern als Spalten
data = data.dropna()

# 2. Portfolio Erstellung (60% S&P 500, 40% Long Term Bonds)
returns = data.pct_change().dropna()

# Direkter Zugriff auf die Spaltennamen
weights = np.array([cfg.portfolio.weight_equity, cfg.portfolio.weight_bonds])
portfolio_returns = (returns[["^GSPC", "VUSTX"]] * weights).sum(axis=1)

df = pd.DataFrame(index=portfolio_returns.index)

# Einzel-Returns von S&P 500 und Bonds
df['Returns_GSPC'] = returns['^GSPC']
df['Returns_VUSTX'] = returns['VUSTX']

df['Returns'] = portfolio_returns
df['Cumulative_Returns'] = (1 + df['Returns']).cumprod()

# --- CASH-RENDITE INTEGRATION ---
# ^IRX gibt die jährliche Rendite in % an. Umrechnung in tägliche Rendite:
# (Wert / 100) / 252 Handelstage
df['Cash_Returns'] = (data['^IRX'] / 100) / 252

df['VIX'] = data['^VIX']
df['TNX_10Y'] = data['^TNX']
df['IRX_3M'] = data['^IRX']

df = df.dropna()
print(f"Erfolgreich geladen: {df.index[0].date()} bis {df.index[-1].date()}")

print(df)

In [ ]:
from pathlib import Path

output_path = Path(cfg.data_path("preprocessed"))

# Verzeichnis anlegen, wenn nicht vorhanden
output_path.parent.mkdir(exist_ok=True)

# Speichern als Parquet
df.to_parquet(output_path)
print(f"Dataframe erfolgreich unter {output_path} gespeichert.")